# Day 19: MatMul Tiling

一个 program 计算 BM×BN 输出块，沿 K 分 BK 累积。边缘 tile 读补零、写加 mask。变一个 tile 参数比较，不把更大 tile 必定更快当成结论。

参考源码：`learning/kernels/triton_ops.py`。

每日概念、误区、面试问答：[知识讲义](../docs/KNOWLEDGE_DAY15_21.md)。

当前个人进度以[进度记录](../docs/LEARNING_PROGRESS.md)为准；运行参考答案不等于已独立掌握。

[Day 19完整讲义](../docs/DAY19_TILED_MATMUL.md)：概念、计算契约、源码实验、调试优化、答案与资料。

先写自己的预测，再执行原有代码单元格；不要自动勾选个人验收。

## 1. 图解与数字例子

把矩阵乘看成小块积木：

```text
A [M,K] × B [K,N] → C [M,N]
每个program负责C的一块BM×BN
沿K轴反复搬A/B小块 → acc += A_tile @ B_tile
```

M=65、N=49、K=33，tile=32时，输出grid=3×2。
K循环两次，第二次只有1个有效K元素，其余补0。
最后M tile只剩1行，最后N tile剩17列；必须分别mask读取与写出。

## 先预测

执行前写下预期 shape、输出或调度顺序；在下面另建 Markdown 单元格保存自己的答案，不修改参考实现作为唯一证据。

In [ ]:
import json
import os
import sys
from pathlib import Path
root = Path.cwd()
if root.name == 'learning':
    root = root.parent
if not (root / 'learning/curriculum.py').is_file():
    raise RuntimeError('Open from repository root or learning directory.')
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
import torch
from learning.curriculum import run_day
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)


In [ ]:
report = run_day(19, DEVICE)
print(json.dumps(report, indent=2, ensure_ascii=False))
destination = Path('logs') / f"day{report['day']:02d}-course.json"
destination.parent.mkdir(parents=True, exist_ok=True)
destination.write_text(json.dumps(report, indent=2, ensure_ascii=False) + '\n')
print('Report:', destination)


## 自测与失败实验

M/N/K 不是 tile 的整数倍时哪三处要 mask？

记录你改变的唯一变量、预期、实际结果和差异原因。

`requires_cuda` / `requires_cuda_compiler` / `external_service_not_configured` 都不是通过。

## 5. 自测与参考答案

**问：tile=32时65×49要几块？**
答：ceil(65/32)×ceil(49/32)=3×2，不是向下取整。

**问：为什么K尾部补0？**
答：对矩阵乘求和0是中性贡献，越界数据不能参与。

**问：tile越大越快吗？**
答：不一定，资源占用、并行度和尾部利用率会改变。

**问：与cuBLAS一个小例子相比更快，能替代它吗？**
答：不能，要覆盖支持范围、精度、规模和多种硬件，本课只是教学kernel。

完整实现限制、调试步骤与资料见[本日讲义](../docs/DAY19_TILED_MATMUL.md)。

以上是参考答案，不是个人学习结果。

## 验收记录

- [ ] 能解释原理
- [ ] 能自行复现
- [ ] 能解释一次失败
- [ ] 能说明当前实现不支持什么

个人笔记：

待填写。